In [2]:
import kagglehub
import pandas as pd
import json
import random

# Download dataset
path = kagglehub.dataset_download("gauravtopre/bank-customer-churn-dataset")
print(f"Dataset downloaded to: {path}")

# Load dataset
df = pd.read_csv(f"{path}/Bank Customer Churn Prediction.csv")
print(f"Loaded {len(df)} rows with {len(df.columns)} columns")

# Basic validation
assert 'churn' in df.columns, "Missing 'churn' column"
assert 'gender' in df.columns, "Missing 'gender' column"
print(f"Churn distribution: {df['churn'].value_counts().to_dict()}")

random.seed(42)

def row_to_text(row):
    pronoun = "he" if row['gender'].lower() == 'male' else "she"
    pronoun_cap = pronoun.capitalize()
    possessive = "his" if row['gender'].lower() == 'male' else "her"
    
    text = (
        f"This bank customer is a {row['age']}-year-old {row['gender'].lower()} from {row['country']}. "
        f"{pronoun_cap} has a credit score of {row['credit_score']} and has been with the bank for {row['tenure']} years. "
        f"The customer's account balance is ${row['balance']:.2f} and {pronoun} uses {row['products_number']} product(s) from the bank. "
        f"{pronoun_cap} {'has' if row['credit_card'] == 1 else 'does not have'} a credit card and "
        f"{'is' if row['active_member'] == 1 else 'is not'} an active member. "
        f"The estimated salary for this customer is ${row['estimated_salary']:.2f}."
    )
    return text

# Transform to JSON format
data = []
for idx, row in df.iterrows():
    data.append({
        "text": row_to_text(row),
        "class": int(row['churn'])
    })
    if idx == 0:
        print(f"\nSample entry:\n{json.dumps(data[0], indent=2)}")

# Add train/test split (80/20)
for entry in data:
    entry["split"] = "train" if random.random() < 0.8 else "test"

# Save to JSON
output_file = "/Users/b715wt/ACT/Deep_Agents_Network/tests/bank_churn_dataset.json"
with open(output_file, 'w') as f:
    json.dump(data, f, indent=2)

print(f"\n✓ Saved {len(data)} entries to {output_file}")
print(f"  Class 0 (no churn): {sum(1 for d in data if d['class'] == 0)}")
print(f"  Class 1 (churn): {sum(1 for d in data if d['class'] == 1)}")

# Print split statistics
train_count = sum(1 for d in data if d['split'] == 'train')
test_count = sum(1 for d in data if d['split'] == 'test')
print(f"\nSplit: {train_count} train, {test_count} test")

# Print distirbution
train_count_churn = sum(1 for d in data if (d['split'] == 'train' and d['class']==1))
train_count_nochurn = sum(1 for d in data if (d['split'] == 'train' and d['class']==0))
test_count_churn = sum(1 for d in data if (d['split'] == 'test' and d['class']==1))
test_count_nochurn = sum(1 for d in data if (d['split'] == 'test' and d['class']==0))
print(f"\nSplit train: {train_count_churn} churn, {train_count_nochurn} no churn")
print(f"\nSplit test: {test_count_churn} churn, {test_count_nochurn} no churn")

Dataset downloaded to: /Users/b715wt/.cache/kagglehub/datasets/gauravtopre/bank-customer-churn-dataset/versions/1
Loaded 10000 rows with 12 columns
Churn distribution: {0: 7963, 1: 2037}

Sample entry:
{
  "text": "This bank customer is a 42-year-old female from France. She has a credit score of 619 and has been with the bank for 2 years. The customer's account balance is $0.00 and she uses 1 product(s) from the bank. She has a credit card and is an active member. The estimated salary for this customer is $101348.88.",
  "class": 1
}

✓ Saved 10000 entries to /Users/b715wt/ACT/Deep_Agents_Network/tests/bank_churn_dataset.json
  Class 0 (no churn): 7963
  Class 1 (churn): 2037

Split: 8031 train, 1969 test

Split train: 1652 churn, 6379 no churn

Split test: 385 churn, 1584 no churn
